# Task 6: MLOps and Deployment Simulation

Simulates moving the traffic-volume regressor from experimentation to a (mocked) deployment:

- **6.1 Model versioning** — train and document three candidate versions, on a *chronological* train/holdout split (not the random split used in Task 1 — see the rationale below).
- **6.2 Experiment tracking** — log every version's parameters, metrics, and model artifact to MLflow, and register the chosen "production" version in the MLflow Model Registry.
- **6.3 Deployment** — a real FastAPI app (`deployment_api.py`, alongside this notebook) serving the production model behind a `/predict` endpoint, demonstrated here with `TestClient` (no server process needed).
- **6.4 Monitoring** — check for feature-distribution drift (Kolmogorov-Smirnov test) and prediction-error drift (MAE vs. a stored reference), against three scenarios: an in-distribution sanity check, real data from a year later, and a synthetic stress test.
- **6.5 Alerting** — a PASS/ALERT status per check, aggregated into an overall system status and rendered as a small dashboard table.

**Files this notebook produces** (all read by `deployment_api.py`): `production_model.joblib`, `model_metadata.json`, `reference_stats.json`, `model_registry.json`. Run this notebook from the same directory as `deployment_api.py` so both can find each other.

## 1. Imports

In [1]:
import json
import logging
import pathlib
import sys

import joblib
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

RANDOM_STATE = 42

## 2. Logging setup

Same dual-handler pattern used throughout — DEBUG+ to `pipeline.log`, INFO+ inline — with the re-run guard notebooks need (a cell can be executed more than once per kernel session).

In [2]:
logger = logging.getLogger("task6_mlops")


def setup_logging():
    if logger.handlers:
        return
    logger.setLevel(logging.DEBUG)
    fmt = logging.Formatter(
        "%(asctime)s | %(levelname)s | %(name)s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )

    fh = logging.FileHandler("pipeline.log", mode="a", encoding="utf-8")
    fh.setLevel(logging.DEBUG)
    fh.setFormatter(fmt)

    ch = logging.StreamHandler()
    ch.setLevel(logging.INFO)
    ch.setFormatter(fmt)

    logger.addHandler(fh)
    logger.addHandler(ch)
    logger.propagate = False


setup_logging()
logger.info("Logging initialised")

2026-09-24 22:18:26 | INFO | task6_mlops | Logging initialised


## 3. Load data and rebuild the feature set

Task 6 deploys **one** model — the traffic-volume regressor — so only that target is needed here (unlike Task 1, `accident_risk` isn't built). Rows are sorted by `date_time`: sections 6.1 and 6.4 both depend on the data being in chronological order.

In [3]:
FEATURES_PATH = "Features_Metro_Interstate_Traffic_Volume.csv"

df = pd.read_csv(FEATURES_PATH, parse_dates=["date_time"])
df = df.sort_values("date_time").reset_index(drop=True)
logger.info("Loaded %s rows, %s columns from %s", len(df), df.shape[1], FEATURES_PATH)

is_holiday = df["holiday"].notna() & (df["holiday"] != "None")
df["is_holiday"] = is_holiday.astype(int)

time_features = [
    "hour", "day_of_week", "is_weekend",
    "hour_sin", "hour_cos", 
]
exclude_raw_text = {"weather_main", "weather_description"}
weather_onehot = sorted(
    c for c in df.columns if c.startswith("weather_") and c not in exclude_raw_text
)
weather_derived = [
    "is_precipitation", "total_precipitation", "is_overcast",
    "is_clear", "is_severe_weather", "clouds_all", "temp_c",
]
feature_cols = time_features + weather_onehot + weather_derived + ["is_holiday"]

model_df = df[["date_time"] + feature_cols + ["traffic_volume"]].dropna().reset_index(drop=True)
logger.info("Modelling on %s rows, %s features", len(model_df), len(feature_cols))


def map_weather_categories(df, weather_onehot):
    """{category name -> its one-hot column}, discovered from the data (see Task 5) rather than assumed."""
    mapping = {}
    for category in sorted(df["weather_main"].dropna().unique()):
        subset = df.loc[df["weather_main"] == category, weather_onehot]
        matches = subset.columns[(subset == 1).all(axis=0)]
        if len(matches) == 1:
            mapping[str(category)] = matches[0]
    return mapping


weather_category_to_column = map_weather_categories(df, weather_onehot)
logger.info("Weather categories detected: %s", sorted(weather_category_to_column))

2026-09-24 22:18:27 | INFO | task6_mlops | Loaded 48187 rows, 38 columns from Features_Metro_Interstate_Traffic_Volume.csv
2026-09-24 22:18:27 | INFO | task6_mlops | Modelling on 48187 rows, 24 features
2026-09-24 22:18:27 | INFO | task6_mlops | Weather categories detected: ['Clear', 'Clouds', 'Drizzle', 'Fog', 'Haze', 'Mist', 'Rain', 'Smoke', 'Snow', 'Squall', 'Thunderstorm']


## 6.1 Model versioning

**Why a chronological split, not Task 1's random split:** a model destined for deployment will only ever predict the *future* relative to its training data. A random 80/20 split lets the model implicitly "see" patterns from dates after the ones it's tested on, which flatters the metric but doesn't reflect how the model will actually be used. Splitting by date — train on the earlier 80%, evaluate on the later 20% — is the standard practice for a production-bound time series model, and this same split doubles as the reference/current split for monitoring in section 6.4.

Three candidate versions are trained and compared on the same holdout:
- **v1** — Linear Regression (the Task 1 baseline)
- **v2** — Random Forest, Task 1's chosen configuration (`n_estimators=200, max_depth=12`)
- **v3** — a deeper Random Forest candidate (`n_estimators=300, max_depth=18`), to check whether more capacity is actually worth it

In [4]:
split_idx = int(len(model_df) * 0.8)
train_df = model_df.iloc[:split_idx]
holdout_df = model_df.iloc[split_idx:]
logger.info(
    "Chronological split: train %s rows (%s to %s), holdout %s rows (%s to %s)",
    len(train_df), train_df["date_time"].min(), train_df["date_time"].max(),
    len(holdout_df), holdout_df["date_time"].min(), holdout_df["date_time"].max(),
)

X_train, y_train = train_df[feature_cols], train_df["traffic_volume"]
X_hold, y_hold = holdout_df[feature_cols], holdout_df["traffic_volume"]

versions = {
    "v1": ("Linear Regression baseline", LinearRegression(), {}),
    "v2": (
        "Random Forest (Task 1 configuration)",
        RandomForestRegressor(n_estimators=200, max_depth=12, random_state=RANDOM_STATE, n_jobs=-1),
        {"n_estimators": 200, "max_depth": 12},
    ),
    "v3": (
        "Random Forest (deeper candidate)",
        RandomForestRegressor(n_estimators=300, max_depth=18, random_state=RANDOM_STATE, n_jobs=-1),
        {"n_estimators": 300, "max_depth": 18},
    ),
}

registry_rows = []
fitted_models = {}
for vid, (description, model, params) in versions.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_hold)
    mae = mean_absolute_error(y_hold, preds)
    r2 = r2_score(y_hold, preds)
    fitted_models[vid] = model
    registry_rows.append(
        {"version": vid, "description": description, "params": params, "MAE": round(mae, 2), "R2": round(r2, 4)}
    )
    logger.info("[Versioning] %s (%s) -> MAE %.2f, R2 %.4f", vid, description, mae, r2)

registry_df = pd.DataFrame(registry_rows)
registry_df

2026-09-24 22:18:27 | INFO | task6_mlops | Chronological split: train 38549 rows (2012-10-02 09:00:00 to 2017-11-01 19:00:00), holdout 9638 rows (2017-11-01 20:00:00 to 2018-09-30 23:00:00)
2026-09-24 22:18:27 | INFO | task6_mlops | [Versioning] v1 (Linear Regression baseline) -> MAE 829.41, R2 0.7117
2026-09-24 22:18:32 | INFO | task6_mlops | [Versioning] v2 (Random Forest (Task 1 configuration)) -> MAE 292.03, R2 0.9283
2026-09-24 22:18:41 | INFO | task6_mlops | [Versioning] v3 (Random Forest (deeper candidate)) -> MAE 308.77, R2 0.9235


,version,description,params,MAE,R2
0,v1,Linear Regression baseline,{},829.41,0.7117
1,v2,Random Forest (Task 1 configuration),"{'n_estimators': 200, 'max_depth': 12}",292.03,0.9283
2,v3,Random Forest (deeper candidate),"{'n_estimators': 300, 'max_depth': 18}",308.77,0.9235


In [5]:
# Pick the production version: lowest holdout MAE, but prefer the simpler/cheaper model
# when a more complex one only improves MAE marginally (< 1%) - added capacity that doesn't
# clearly pay for itself isn't worth the extra training/serving cost or overfitting risk.
best = registry_df.sort_values("MAE").iloc[0]
v2_mae = registry_df.set_index("version").loc["v2", "MAE"]
if best["version"] == "v3" and (v2_mae - best["MAE"]) / v2_mae < 0.01:
    production_version = "v2"
    decision_reason = "v3 improves MAE by <1% over v2; the added complexity isn't worth it"
else:
    production_version = best["version"]
    decision_reason = f"lowest holdout MAE ({best['MAE']})"

print(f"Production version: {production_version} — {decision_reason}")
logger.info("Selected production version %s: %s", production_version, decision_reason)

registry_df.to_json("model_registry.json", orient="records", indent=2)
logger.info("Wrote model_registry.json (%s versions documented)", len(registry_df))

2026-09-24 22:18:41 | INFO | task6_mlops | Selected production version v2: lowest holdout MAE (292.03)
2026-09-24 22:18:41 | INFO | task6_mlops | Wrote model_registry.json (3 versions documented)


Production version: v2 — lowest holdout MAE (292.03)


## 6.2 Experiment tracking (MLflow)

Same robust local-store setup as Task 4 — an **absolute** `file:` URI built with `Path.resolve().as_uri()` and the folder created before use, which avoids the relative-URI resolution failures that a relative `"file:./mlruns"` string can trigger. Each version is logged as its own run (parameters, metrics, and the model artifact), and `mlflow.register_model` adds it to the **Model Registry** under one registered model name — so all three versions are tracked as versions of the *same* deployable model, not three unrelated experiments.

In [6]:
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient
import os
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"

mlruns_dir = pathlib.Path("mlruns").resolve()
mlruns_dir.mkdir(exist_ok=True)
mlflow.set_tracking_uri(mlruns_dir.as_uri())
mlflow.set_experiment("metro_traffic_mlops_versions")
logger.info("MLflow experiment set: metro_traffic_mlops_versions (tracking URI: %s)", mlruns_dir.as_uri())

REGISTERED_MODEL_NAME = "metro_traffic_volume_regressor"
registry_version_numbers = {}

for vid, (description, model, params) in versions.items():
    with mlflow.start_run(run_name=f"traffic_volume_{vid}") as run:
        mlflow.set_tag("version_label", vid)
        mlflow.set_tag("task", "regression")
        mlflow.log_param("model_type", description)
        for k, v in params.items():
            mlflow.log_param(k, v)

        preds = fitted_models[vid].predict(X_hold)
        mae = mean_absolute_error(y_hold, preds)
        r2 = r2_score(y_hold, preds)
        mlflow.log_metric("MAE", mae)
        mlflow.log_metric("R2", r2)

        mlflow.sklearn.log_model(fitted_models[vid], "model")
        model_uri = f"runs:/{run.info.run_id}/model"
        model_version = mlflow.register_model(model_uri, REGISTERED_MODEL_NAME)
        registry_version_numbers[vid] = model_version.version

        logger.info(
            "[MLflow] %s (%s) -> MAE %.2f, R2 %.4f, registry version %s",
            vid, description, mae, r2, model_version.version,
        )

2026-09-24 22:18:45 | INFO | task6_mlops | MLflow experiment set: metro_traffic_mlops_versions (tracking URI: file:///C:/Projects/Learning/NUS_AI_ML_Data_Science_Programme/33rd_week_capstone_project/smart_traffic_capstone_project/smart_traffic_capstone_project/part3_machine_learning/notebooks/mlruns)
2026/09/24 22:18:46 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Successfully registered model 'metro_traffic_volume_regressor'.
2026/09/24 22:19:16 WARNING mlflow.tracking._model_registry.fluent: Run with id ea9a2475f39f443499a048da350f4717 has no artifacts at artifact path 'model', registering model based on models:/m-fc332051c9be41be94aeb04a310ef0e5 instead
Created version '1' of model 'metro_traffic_volume_regressor'.
2026-09-24 22:19:16 | INFO | task6_mlops | [MLflow] v1 (Linear Regression baseline) -> MAE 829.41, R2 0.7117, registry version 1
2026/09/24 22:19:16 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` ins

In [7]:
# Mark the chosen version as the one actually served. MLflow's stage-based registry
# (transition_model_version_stage) is being superseded by alias-based promotion in newer
# releases - try stages first, and fall back to an alias so this works either way.
client = MlflowClient()
prod_registry_version = registry_version_numbers[production_version]

try:
    client.transition_model_version_stage(
        name=REGISTERED_MODEL_NAME, version=prod_registry_version, stage="Production"
    )
    logger.info("Marked registry version %s as stage=Production", prod_registry_version)
except Exception as exc:
    client.set_registered_model_alias(REGISTERED_MODEL_NAME, "production", prod_registry_version)
    logger.info(
        "Stage-based promotion unavailable (%s); set alias 'production' -> version %s instead",
        exc, prod_registry_version,
    )

C:\Users\krmee\AppData\Local\Temp\ipykernel_33884\224249696.py:8: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(
2026-09-24 22:20:06 | INFO | task6_mlops | Marked registry version 2 as stage=Production


In [8]:
# Pull every tracked run back from MLflow - confirms params/metrics/tags were all recorded,
# without needing to leave the notebook (mlflow ui gives the fuller interactive dashboard).
runs = mlflow.search_runs(experiment_names=["metro_traffic_mlops_versions"])
display_cols = [
    c for c in ["tags.version_label", "params.model_type", "metrics.MAE", "metrics.R2", "run_id"]
    if c in runs.columns
]
runs[display_cols].sort_values("tags.version_label")

,tags.version_label,params.model_type,metrics.MAE,metrics.R2,run_id
2,v1,Linear Regression baseline,829.414326,0.711745,ea9a2475f39f443499a048da350f4717
3,v1,Linear Regression baseline,829.414326,0.711745,5985e289d0694df4a3b4b10e2fccfb37
1,v2,Random Forest (Task 1 configuration),292.030189,0.928337,f5aef31ac5a946e286ef418fd092786e
0,v3,Random Forest (deeper candidate),308.773770,0.923540,833d30f1e3fe42bea1a60b8eb69b5d94


## Bridge: build the deployment artifacts

Same reasoning as Task 5: the production model is **refit on the full dataset** rather than served straight from the holdout-trained version above — the versioning comparison already happened, so the deployed copy should use every available row. `deployment_api.py` doesn't read the MLflow store directly (a deployed API shouldn't need the whole tracking store just to serve predictions); instead it reads three small, purpose-built files written here: the model itself, its metadata/feature contract, and a monitoring reference baseline.

In [9]:
prod_description, prod_template, prod_params = versions[production_version]
if isinstance(prod_template, LinearRegression):
    production_model = LinearRegression()
else:
    production_model = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1, **prod_params)

production_model.fit(model_df[feature_cols], model_df["traffic_volume"])
joblib.dump(production_model, "production_model.joblib")
logger.info("Saved production_model.joblib (version %s, refit on all %s rows)", production_version, len(model_df))

reference_mae = float(registry_df.set_index("version").loc[production_version, "MAE"])
metadata = {
    "model_version": production_version,
    "model_description": prod_description,
    "feature_cols": feature_cols,
    "weather_category_to_column": weather_category_to_column,
    "holdout_MAE": reference_mae,
    "holdout_R2": float(registry_df.set_index("version").loc[production_version, "R2"]),
}
pathlib.Path("model_metadata.json").write_text(json.dumps(metadata, indent=2))
logger.info("Wrote model_metadata.json")

2026-09-24 22:20:12 | INFO | task6_mlops | Saved production_model.joblib (version v2, refit on all 48187 rows)
2026-09-24 22:20:12 | INFO | task6_mlops | Wrote model_metadata.json


In [10]:
# Reference window for monitoring: the most recent ~12 months of the TRAIN period, not the
# whole multi-year training history. Comparing a same-sized recent slice to another recent
# slice keeps the comparison season-matched - comparing against 5 years of history would flag
# ordinary seasonality as "drift" on almost every check.
MONITORED_FEATURES = ["temp_c", "clouds_all", "total_precipitation"]
rng = np.random.RandomState(RANDOM_STATE)

ref_window_start = train_df["date_time"].max() - pd.DateOffset(months=12)
ref_window_df = train_df[train_df["date_time"] >= ref_window_start]

# Split the reference window itself into two disjoint halves: one to build the stored
# reference sample, one held out purely as an in-distribution sanity check (Demo A, 6.4).
shuffled_idx = ref_window_df.sample(frac=1.0, random_state=RANDOM_STATE).index
half = len(shuffled_idx) // 2
ref_pool_df = ref_window_df.loc[shuffled_idx[:half]]
insample_check_df = ref_window_df.loc[shuffled_idx[half:]]

reference_samples = {}
for feat in MONITORED_FEATURES:
    vals = ref_pool_df[feat].dropna().values
    sample = rng.choice(vals, size=min(500, len(vals)), replace=False)
    reference_samples[feat] = sample.tolist()

reference_stats = {
    "reference_period": {
        "start": str(ref_window_df["date_time"].min()),
        "end": str(ref_window_df["date_time"].max()),
    },
    "reference_mae": reference_mae,
    "feature_samples": reference_samples,
    "thresholds": {"ks_pvalue": 0.05, "mae_ratio": 1.3},
}
pathlib.Path("reference_stats.json").write_text(json.dumps(reference_stats, indent=2))
logger.info("Wrote reference_stats.json (reference window: %s to %s)", reference_stats["reference_period"]["start"], reference_stats["reference_period"]["end"])

2026-09-24 22:20:13 | INFO | task6_mlops | Wrote reference_stats.json (reference window: 2016-11-01 19:00:00 to 2017-11-01 19:00:00)


## 6.3 Deployment

`deployment_api.py` (in the same directory as this notebook) is a real FastAPI app — `/health`, `/model/info`, `POST /predict`, and `POST /monitoring/status`. It loads the three files just written and builds the same engineered feature row internally that Task 1/4/5 use, from a small, human-friendly JSON contract (hour, day of week, weather condition, temperature, cloud cover, precipitation), so a caller never needs to know about one-hot columns or cyclical encodings.

`TestClient` drives the FastAPI app in-process — no `uvicorn` server or port needed, which keeps this notebook's demo fully offline and deterministic. To actually serve it, run `uvicorn deployment_api:app --reload` from a terminal in this directory instead.

In [12]:
sys.path.insert(0, ".")
import deployment_api
from fastapi.testclient import TestClient

client = TestClient(deployment_api.app)

print("GET /health ->", client.get("/health").json())
print("GET /model/info ->", client.get("/model/info").json())

GET /health -> {'status': 'ok', 'model_version': 'v2'}
GET /model/info -> {'model_version': 'v2', 'model_description': 'Random Forest (Task 1 configuration)', 'holdout_MAE': 292.03, 'holdout_R2': 0.9283, 'n_features': 24, 'weather_categories': ['Clear', 'Clouds', 'Drizzle', 'Fog', 'Haze', 'Mist', 'Rain', 'Smoke', 'Snow', 'Squall', 'Thunderstorm']}


C:\Users\krmee\AppData\Local\Programs\Python\Python312\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


In [13]:
sample_request = {
    "hour": 8,
    "day_of_week": 2,
    "is_holiday": False,
    "weather_main": "Clear",
    "temp_c": 18.0,
    "clouds_all": 20.0,
    "rain_1h": 0.0,
    "snow_1h": 0.0,
}
response = client.post("/predict", json=sample_request)
print("POST /predict ->", response.status_code)
print(json.dumps(response.json(), indent=2))

POST /predict -> 200
{
  "predicted_traffic_volume": 5759.7,
  "model_version": "v2",
  "model_description": "Random Forest (Task 1 configuration)",
  "predicted_at": "2026-09-24T14:25:19.525471+00:00"
}


## 6.4 Monitoring

Three scenarios, checked through the same `/monitoring/status` endpoint the deployed API exposes:

- **Demo A — in-distribution sanity check:** a random sample from the *same* reference window used to build the baseline (the other half of the split above). Two random slices of the same period should look statistically identical — the cleanest possible "everything normal" case.
- **Demo B — real data, one year later:** the actual chronological holdout set from section 6.1. This is genuine data, not synthetic, so whatever it shows is a real result worth reading carefully rather than a guaranteed outcome either way.
- **Demo C — synthetic stress test:** Demo B's batch with `temp_c` shifted +15°C, `clouds_all` pushed toward saturation, and the "actual" traffic volumes corrupted with heavy noise — simulating a sensor fault or a genuinely broken model, to confirm the alerting logic actually fires when something is wrong.

In [14]:
def build_monitoring_payload(batch_df, include_actuals=True):
    """DataFrame rows -> the JSON body /monitoring/status expects."""
    records = []
    for _, row in batch_df.iterrows():
        record = {
            "hour": int(row["hour"]),
            "day_of_week": int(row["day_of_week"]),
            "is_holiday": bool(row["is_holiday"]),
            "weather_main": next(
                cat for cat, col in weather_category_to_column.items() if row.get(col, 0) == 1
            ),
            "temp_c": float(row["temp_c"]),
            "clouds_all": float(row["clouds_all"]),
            "rain_1h": 0.0,
            "snow_1h": float(row["total_precipitation"]),
        }
        if include_actuals:
            record["actual_traffic_volume"] = float(row["traffic_volume"])
        records.append(record)
    return {"records": records}

In [16]:
SAMPLE_SIZE = 300

# Demo A: in-distribution sanity check
sample_a = insample_check_df.sample(n=min(SAMPLE_SIZE, len(insample_check_df)), random_state=RANDOM_STATE)

# Demo B: real data, one year later
sample_b = holdout_df.sample(n=min(SAMPLE_SIZE, len(holdout_df)), random_state=RANDOM_STATE)

# Demo C: synthetic stress test, built from Demo B
sample_c = sample_b.copy()
sample_c["temp_c"] = sample_c["temp_c"] + 15
sample_c["clouds_all"] = (sample_c["clouds_all"] + 40).clip(0, 100)
sample_c["traffic_volume"] = sample_c["traffic_volume"] + rng.normal(0, 4000, size=len(sample_c))

scenarios = {
    "A - in-distribution sanity check": sample_a,
    "B - real data, one year later": sample_b,
    "C - synthetic stress test": sample_c,
}

monitoring_results = {}
for label, batch in scenarios.items():
    payload = build_monitoring_payload(batch)
    response = client.post("/monitoring/status", json=payload)
    monitoring_results[label] = response.json()
    print(f"--- {label} ---")
    print(json.dumps(monitoring_results[label], indent=2))
    print()

--- A - in-distribution sanity check ---
{
  "overall_status": "PASS",
  "checks": {
    "feature_drift_temp_c": {
      "status": "PASS",
      "ks_statistic": 0.0507,
      "p_value": 0.70396,
      "detail": "temp_c distribution consistent with reference (KS p=0.7040)"
    },
    "feature_drift_clouds_all": {
      "status": "PASS",
      "ks_statistic": 0.0193,
      "p_value": 1.0,
      "detail": "clouds_all distribution consistent with reference (KS p=1.0000)"
    },
    "feature_drift_total_precipitation": {
      "status": "PASS",
      "ks_statistic": 0.0,
      "p_value": 1.0,
      "detail": "total_precipitation distribution consistent with reference (KS p=1.0000)"
    },
    "prediction_error_drift": {
      "status": "PASS",
      "current_mae": 199.56,
      "reference_mae": 292.03,
      "ratio": 0.683,
      "detail": "current MAE 199.6 vs reference 292.0 (x0.68)"
    }
  },
  "n_records": 300,
  "checked_at": "2026-09-24T14:25:41.978852+00:00"
}

--- B - real data, on

## 6.5 Alerting dashboard

A minimal PASS/ALERT summary — the "simple dashboard" the task asks for — built directly from the `/monitoring/status` responses above, not recomputed separately, so the dashboard can never drift from what the API actually reported.

In [17]:
dashboard_rows = []
for label, result in monitoring_results.items():
    row = {"scenario": label, "overall_status": result["overall_status"]}
    for check_name, check in result["checks"].items():
        row[check_name] = check["status"]
    dashboard_rows.append(row)

dashboard_df = pd.DataFrame(dashboard_rows).set_index("scenario")


def _color_status(value):
    if value == "ALERT":
        return "background-color: #F8D7DA; color: #842029; font-weight: 600"
    if value == "PASS":
        return "background-color: #D1E7DD; color: #0F5132; font-weight: 600"
    return ""


styler = dashboard_df.style
# pandas >=2.1 renamed Styler.applymap to Styler.map; support either
styled = styler.map(_color_status) if hasattr(styler, "map") else styler.applymap(_color_status)
styled

,overall_status,feature_drift_temp_c,feature_drift_clouds_all,feature_drift_total_precipitation,prediction_error_drift
scenario,,,,,
A - in-distribution sanity check,PASS,PASS,PASS,PASS,PASS
"B - real data, one year later",ALERT,ALERT,PASS,PASS,PASS
C - synthetic stress test,ALERT,ALERT,ALERT,PASS,ALERT
